### Title: 00e_text_matching_fda_fndds
### Purpose: Text match FNDDS ingredient descriptions in all NHANES participants to FDA
### Date: July 2, 2025
### Author: Jules Larke

In [1]:
import pandas as pd
import string
import re
import nltk
nltk.data.path.append('/Users/jules.larke/nltk_data')
wn = nltk.WordNetLemmatizer()

In [2]:
from polyfuzz.models import TFIDF
from polyfuzz import PolyFuzz

In [ ]:
# all_ingred_list.csv is a list of all unique ingredients in the all NHANES participants. This was generated from script 00a (prior to excluding participants with <XX% of kcal consumed mappable to the DFG2)
fda = pd.read_csv('../../data/00/ingredient_matching/wweia_fda_unique_ingredients.csv')

In [4]:
fda.rename(columns={'Basic Ingredient Description': 'fda_desc'}, inplace=True)

In [5]:
fda['fda_desc'] = fda['fda_desc'].str.replace('_', ' ')

In [6]:
# load the FNDDS full ingredient list (compiled from 2001-2018)
fndds = pd.read_csv('../../data/00/ingredient_matching/fndds_all_ingredient_nutrient_values_2023.csv', usecols=['Ingredient code', 'Ingredient description'])

In [7]:
fndds['Ingredient description'] = fndds['Ingredient description'].str.replace('_', ' ')

In [8]:
fndds.rename(columns={'Ingredient description': 'fndds_desc'}, inplace=True)

In [9]:
punct = string.punctuation[0:11] + string.punctuation[13:] # remove '-' from the list of punctuation.
stopwords = ['','and', 'to', 'not', 'no',  'bkdfrd', 'ppd', 'pkgddeli', 'pkgd', 'xtra', 'oz', 'in', 'with', 'or', 'only', 'cooking', 'as', 'food', 'distribution', 'usda', 'form', 'w', 'wo', 'ns', 'nfs', 'incl']

def clean_text(text):
    text = "".join([word for word in text if word not in punct])
    tokens = re.split(r'[-\W+]', text)
    text = [word for word in tokens if word not in stopwords]
    text = [wn.lemmatize(word) for word in tokens if word not in stopwords]
    return "default" if text is [] else ' '.join(set(text))

fda['fda_clean'] = fda['fda_desc'].apply(lambda x: clean_text(x.lower()))
fndds['fndds_clean'] = fndds['fndds_desc'].apply(lambda x: clean_text(x.lower()))

In [10]:
fda_tokens = fda['fda_clean'].to_list()
fndds_tokens = fndds['fndds_clean'].to_list()

In [11]:
tfidf = TFIDF(n_gram_range=(1, 3))
model = PolyFuzz(tfidf).match(fda_tokens, fndds_tokens)

In [12]:
match = model.get_matches()
match.rename(columns={'From':'fda_clean', 'To':'fndds_clean'},inplace=True)

In [13]:
matched_1 = match.merge(fndds, on='fndds_clean', how='left')
matched_1.drop_duplicates(subset='fndds_clean')

,fda_clean,fndds_clean,Similarity,Ingredient code,fndds_desc
0,fluid 1 milk fat,d milkfat 1 milk a vitamin added lowfat fluid,0.710,1082,"Milk, lowfat, fluid, 1% milkfat, with added vi..."
1,fluid 2 milk fat,d milkfat milk fat 2 a vitamin added fluid red...,0.703,1079,"Milk, reduced fat, fluid, 2% milkfat, with add..."
2,fluid nonfat milk,d milk fat free a nonfat vitamin added skim fluid,0.720,1085,"Milk, nonfat, fluid, with added vitamin A and ..."
3,whole fluid milk,whole buttermilk fluid milk,0.807,1230,"Milk, buttermilk, fluid, whole"
8,dried nonfat milk,d milk a nonfat vitamin added dry regular,0.580,1154,"Milk, dry, nonfat, regular, with added vitamin..."
...,...,...,...,...,...
918,seaweed raw kombu,seaweed raw agar,0.585,11442,"Seaweed, agar, raw"
921,duck egg cooked,whole duck raw fresh egg,0.587,1138,"Egg, duck, whole, fresh, raw"
922,cooked egg goose,egg whole goose fresh raw,0.705,1139,"Egg, goose, whole, fresh, raw"
926,raab raw broccoli,raab raw broccoli,1.000,11096,"Broccoli raab, raw"


In [14]:
matched_2 = matched_1.merge(fda, on=['fda_clean'], how='left')

In [15]:
matched_2 = matched_2[['fda_desc', 'fndds_desc', 'Ingredient code', 'Similarity']]

In [16]:
matched_2.sort_values(by='fda_desc').to_csv('../../data/00/ingredient_matching/fda_fndds_map.csv', index=None)

## fda_fndds_map_2023.csv will undergo manual curation to determine appropriateness of matches and edits for non-appropriate matches 
## Output file following manual matching: fda_fndds_map_complete.csv